# Week 4 Exercise — Docstring Generator 

**Requirement:** Use an LLM to generate docstrings for Python code, using the same multi-model Gradio

**skills demonstrated:**
- Multi-provider setup (OpenAI, Anthropic, Gemini, Grok, Groq, Ollama, OpenRouter)
- Structured system + user prompts for code generation
- Gradio UI with gr.Code, model dropdown, themed styling
- Compare docstring quality across different models

In [ ]:

import os
import sys

cwd = os.getcwd()
if "week4" in cwd:
    week4_root = cwd if "community-contributions" not in cwd else os.path.dirname(os.path.dirname(cwd))
else:
    week4_root = os.path.join(cwd, "week4")
if week4_root not in sys.path:
    sys.path.insert(0, week4_root)

In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists, begins {openai_api_key[:8]}...")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists, begins {anthropic_api_key[:7]}...")
else:
    print("Anthropic API Key not set (optional)")

if google_api_key:
    print(f"Google API Key exists, begins {google_api_key[:2]}...")
else:
    print("Google API Key not set (optional)")

if grok_api_key:
    print(f"Grok API Key exists, begins {grok_api_key[:4]}...")
else:
    print("Grok API Key not set (optional)")

if groq_api_key:
    print(f"Groq API Key exists, begins {groq_api_key[:4]}...")
else:
    print("Groq API Key not set (optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists, begins {openrouter_api_key[:6]}...")
else:
    print("OpenRouter API Key not set (optional)")

In [ ]:
# Connect to client libraries

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key or "dummy", base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key or "dummy", base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key or "dummy", base_url=grok_url)
groq = OpenAI(api_key=groq_api_key or "dummy", base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openai_api_key, base_url=openrouter_url)
openai = OpenAI(api_key=openai_api_key, base_url=openrouter_url)

In [ ]:
# Models and clients

models = [
    "gpt-4o-mini",
    "gpt-4o",
    "claude-sonnet-4-5-20250929",
    "claude-3-5-haiku-20241022",
    "gemini-2.0-flash",
    "gemini-2.5-pro",
    "grok-4",
    "openai/gpt-oss-120b",
    "qwen2.5-coder",
    "deepseek-coder-v2",
    "qwen/qwen3-coder-30b-a3b-instruct",
]

clients = {
    "gpt-4o-mini": openai,
    "gpt-4o": openai,
    "claude-sonnet-4-5-20250929": anthropic,
    "claude-3-5-haiku-20241022": anthropic,
    "gemini-2.0-flash": gemini,
    "gemini-2.5-pro": gemini,
    "grok-4": grok,
    "openai/gpt-oss-120b": groq,
    "qwen2.5-coder": ollama,
    "deepseek-coder-v2": ollama,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
}

# Use only models with valid clients (OpenAI default)
models = [m for m in models if m in clients]

In [ ]:
# Docstring prompts (Google-style)

system_prompt = """
You are an expert Python developer. Your task is to add comprehensive Google-style docstrings to every function, method, and class in the provided code.

Rules:
- Preserve ALL original code exactly. Do not rename, refactor, or remove anything.
- Add a module-level docstring at the top if the file has no top-level docstring.
- For each function/method: document purpose, Args (param, type, description), Returns, Raises (if applicable).
- For classes: document the class purpose and __init__ parameters.
- Use Google-style format: Args:, Returns:, Raises: sections.
- Return ONLY the complete Python code with docstrings added. No markdown fences or explanation.
"""


def user_prompt_for(code: str) -> str:
    return f"""Add Google-style docstrings to this Python code. Return only the modified code.

```python
{code}
```
"""

In [ ]:
def messages_for(code: str):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(code)},
    ]


def generate_docstrings(model: str, code: str) -> str:
    if model not in clients:
        return f"Model {model} not available. Check API keys."
    client = clients[model]
    try:
        response = client.chat.completions.create(model=model, messages=messages_for(code))
        reply = response.choices[0].message.content
        # Strip markdown code fences if present
        for fence in ("```python", "```py", "```"):
            reply = reply.replace(fence, "").strip()
        return reply
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# Sample code without docstrings

sample_code = """
def fibonacci(n):
    if n <= 0:
        return []
    if n == 1:
        return [0]
    result = [0, 1]
    for i in range(2, n):
        result.append(result[-1] + result[-2])
    return result


def max_subarray_sum(arr):
    max_sum = arr[0]
    current_sum = arr[0]
    for num in arr[1:]:
        current_sum = max(num, current_sum + num)
        max_sum = max(max_sum, current_sum)
    return max_sum


class Calculator:
    def __init__(self, precision=2):
        self.precision = precision

    def add(self, a, b):
        return round(a + b, self.precision)

    def multiply(self, a, b):
        return round(a * b, self.precision)
"""

In [ ]:
# Gradio UI (same pattern as day5: gr.Code, model dropdown, styled)

try:
    from styles import CSS
except ImportError:
    CSS = ""

# Docstring-specific overrides
DOCSTRING_CSS = CSS + """
.input-out textarea { --py-color: #209dd7; }
.output-out textarea { --cpp-color: #2ecc71; }
.output-out textarea {
  background: linear-gradient(180deg, rgba(46,204,113,.18), rgba(46,204,113,.10)) !important;
  border: 1px solid rgba(46,204,113,.35) !important;
  color: rgba(46,204,113,1) !important;
}
""" if CSS else """
:root { --accent: #753991; --card: #161a22; --text: #e9eef5; }
.convert-btn button { background: var(--accent) !important; color: white !important; }
"""

with gr.Blocks(css=DOCSTRING_CSS, theme=gr.themes.Monochrome(), title="Docstring Generator — adams-bolaji") as ui:
    gr.Markdown("## Docstring Generator — Multi-Model")
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            code_in = gr.Code(
                label="Python code (no docstrings)",
                value=sample_code,
                language="python",
                lines=24,
            )
        with gr.Column(scale=6):
            code_out = gr.Code(
                label="Python code (with docstrings)",
                value="",
                language="python",
                lines=24,
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, value=models[0] if models else None, label="Model", show_label=True)
        convert = gr.Button("Generate Docstrings", elem_classes=["convert-btn"])

    convert.click(fn=generate_docstrings, inputs=[model, code_in], outputs=[code_out])

ui.launch(inbrowser=True)